# SageMaker XGBoost Training - AI Bet Prediction Engine (3-Class)

This notebook trains an XGBoost model to predict betting outcomes based on:
- Game statistics and odds
- User emotional state and behavioral patterns
- Historical betting performance
- Team/player performance metrics

## Prediction Classes
- **Class 0**: LOSS (bet lost, stake gone)
- **Class 1**: PUSH (break even, stake returned)
- **Class 2**: WIN (bet won, profit made)

## Prerequisites
1. Upload `training_data_sagemaker_v2.csv` to S3
2. Have AWS credentials configured
3. Run this in SageMaker Studio or Notebook instance

In [ ]:
# Install required packages
!pip install sagemaker boto3 pandas scikit-learn --upgrade

In [ ]:
import sagemaker
import boto3
import pandas as pd
from sagemaker import get_execution_role
from sagemaker.inputs import TrainingInput
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

# Get SageMaker session and role
sagemaker_session = sagemaker.Session()
role = get_execution_role()
region = boto3.Session().region_name

print(f"SageMaker role: {role}")
print(f"Region: {region}")

## 1. Upload Training Data to S3

In [ ]:
# Configuration
bucket = sagemaker_session.default_bucket()  # Or specify your own bucket
prefix = 'serenity-ai-bet-predictor-v2'

# Local file path
local_training_file = 'training_data_sagemaker_v2.csv'

# Upload to S3
s3_training_data = sagemaker_session.upload_data(
    path=local_training_file,
    bucket=bucket,
    key_prefix=f'{prefix}/data'
)

print(f"Training data uploaded to: {s3_training_data}")

## 2. Inspect the Data

In [ ]:
# Load and inspect data
df = pd.read_csv(local_training_file)

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

print("\nLabel distribution:")
label_counts = df['label'].value_counts().sort_index()
print(label_counts)
print("\nClass breakdown:")
print(f"  Class 0 (LOSS): {label_counts.get(0, 0)} samples ({label_counts.get(0, 0)/len(df)*100:.1f}%)")
print(f"  Class 1 (PUSH): {label_counts.get(1, 0)} samples ({label_counts.get(1, 0)/len(df)*100:.1f}%)")
print(f"  Class 2 (WIN):  {label_counts.get(2, 0)} samples ({label_counts.get(2, 0)/len(df)*100:.1f}%)")

print("\nFeature columns:")
print(df.columns.tolist())

print("\nBasic statistics:")
print(df.describe())

## 3. Get XGBoost Container Image

In [ ]:
from sagemaker.image_uris import retrieve

# Get the XGBoost container image
container = retrieve('xgboost', region, version='1.7-1')
print(f"XGBoost container: {container}")

## 4. Configure and Train the Model (Multiclass Classification)

In [ ]:
from sagemaker.estimator import Estimator

# Output path for model artifacts
output_path = f's3://{bucket}/{prefix}/output'

# Create XGBoost estimator
xgb_estimator = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',  # Start with this, scale up if needed
    output_path=output_path,
    sagemaker_session=sagemaker_session
)

# Set hyperparameters for MULTICLASS classification (3 classes)
xgb_estimator.set_hyperparameters(
    objective='multi:softprob',   # Multiclass classification with probabilities
    num_class=3,                  # 3 classes: LOSS(0), PUSH(1), WIN(2)
    num_round=100,                # Number of boosting rounds
    max_depth=5,                  # Maximum tree depth
    eta=0.2,                      # Learning rate
    gamma=4,                      # Minimum loss reduction for split
    min_child_weight=6,           # Minimum sum of weights in child
    subsample=0.8,                # Subsample ratio of training instances
    verbosity=1,                  # Print messages
    eval_metric='mlogloss'        # Multiclass log loss
)

print("Estimator configured for 3-class classification")
print("Classes: 0=LOSS, 1=PUSH, 2=WIN")

In [ ]:
# Define training input
train_input = TrainingInput(
    s3_data=s3_training_data,
    content_type='text/csv'
)

# Start training
print("Starting training job...")
xgb_estimator.fit({'train': train_input})
print("Training completed!")

## 5. Deploy the Model as an Endpoint

In [ ]:
# Deploy the model
xgb_predictor = xgb_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',  # Use smaller instance for inference
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer()
)

print(f"Model deployed to endpoint: {xgb_predictor.endpoint_name}")

## 6. Test the Endpoint (3-Class Predictions)

In [ ]:
import numpy as np

# Test with a sample prediction
# Format: All features except label (50 features total)
# Example: NBA Spread bet, evenly matched teams (likely PUSH)
test_data_push = [
    # sport_nba, sport_nfl, sport_mlb, sport_nhl
    1, 0, 0, 0,
    # bet_type_moneyline, bet_type_spread, bet_type_over_under, bet_type_parlay
    0, 1, 0, 0,
    # bet_amount, budget, budget_percent
    50, 1000, 5,
    # favorite_win_pct, underdog_win_pct, has_key_injuries, bad_weather, is_parlay
    0.51, 0.51, 0, 0, 0,
    # emotional_risk_low, emotional_risk_moderate, emotional_risk_high
    1, 0, 0,
    # emotional_state_neutral, emotional_state_positive, emotional_state_negative, emotional_state_dangerous
    1, 0, 0, 0,
    # chasing_behavior, high_frequency_betting, recent_loss, recent_win
    0, 0, 0, 0,
    # favorite_ppg, underdog_ppg, favorite_fg_pct, underdog_fg_pct
    21.5, 21.5, 0.468, 0.468,
    # favorite_def_rating, underdog_def_rating, home_advantage
    108.5, 108.5, 1,
    # odds_american, implied_probability, confidence_score
    -3.5, 53.5, 55,
    # safety_risk_low, safety_risk_moderate, safety_risk_high
    1, 0, 0,
    # user_experience_novice, user_experience_intermediate, user_experience_expert
    0, 1, 0,
    # emotional_intensity
    2,
    # bias_overconfidence, bias_recency, bias_loss_aversion, bias_confirmation
    0, 0, 0, 0,
    # last_5_win_rate
    0.5,
    # user_age_18_25, user_age_26_35, user_age_36_45, user_age_46_plus
    0, 1, 0, 0
]

# Make prediction
result = xgb_predictor.predict(test_data_push)
probabilities = result  # XGBoost multiclass returns [prob_class0, prob_class1, prob_class2]

print("\n" + "="*60)
print("PREDICTION RESULTS - Evenly Matched Teams (Spread Bet)")
print("="*60)
print(f"Loss Probability:  {probabilities[0]:.2%}")
print(f"Push Probability:  {probabilities[1]:.2%}")
print(f"Win Probability:   {probabilities[2]:.2%}")
print(f"\nPredicted Outcome: {['LOSS', 'PUSH', 'WIN'][np.argmax(probabilities)]}")
print(f"Confidence: {max(probabilities):.2%}")
print("="*60)

In [ ]:
# Test 2: Strong favorite (likely WIN)
test_data_win = [
    1, 0, 0, 0,  # NBA
    1, 0, 0, 0,  # Moneyline
    50, 1000, 5,  # Low risk bet
    0.75, 0.35, 0, 0, 0,  # Strong favorite vs weak underdog
    1, 0, 0,  # Low emotional risk
    1, 0, 0, 0,  # Neutral state
    0, 0, 0, 0,  # No chasing
    26.5, 17.2, 0.495, 0.438,  # Strong stats
    101.5, 116.8, 1,  # Home advantage
    -220, 68.8, 90,  # Heavy favorite odds
    1, 0, 0,  # Low safety risk
    0, 1, 0,  # Intermediate user
    1,  # Low intensity
    0, 0, 0, 0,  # No biases
    0.8,  # Good win rate
    0, 1, 0, 0  # Age 26-35
]

result = xgb_predictor.predict(test_data_win)
probabilities = result

print("\n" + "="*60)
print("PREDICTION RESULTS - Strong Favorite (Moneyline)")
print("="*60)
print(f"Loss Probability:  {probabilities[0]:.2%}")
print(f"Push Probability:  {probabilities[1]:.2%}")
print(f"Win Probability:   {probabilities[2]:.2%}")
print(f"\nPredicted Outcome: {['LOSS', 'PUSH', 'WIN'][np.argmax(probabilities)]}")
print(f"Confidence: {max(probabilities):.2%}")
print("="*60)

In [ ]:
# Test 3: Risky parlay with emotional betting (likely LOSS)
test_data_loss = [
    1, 0, 0, 0,  # NBA
    0, 0, 0, 1,  # Parlay
    200, 400, 50,  # High risk - 50% of budget
    0.52, 0.58, 0, 0, 1,  # Underdog favored, parlay
    0, 0, 1,  # High emotional risk
    0, 0, 1, 0,  # Negative state
    1, 1, 1, 0,  # Chasing, high frequency, recent loss
    18.9, 20.2, 0.448, 0.462,  # Mediocre stats
    112.3, 108.9, 0,  # No home advantage
    250, 28.6, 45,  # Long shot odds
    0, 0, 1,  # High safety risk
    0, 1, 0,  # Intermediate user
    6,  # High intensity
    1, 1, 1, 0,  # Multiple biases
    0.4,  # Poor win rate
    0, 1, 0, 0  # Age 26-35
]

result = xgb_predictor.predict(test_data_loss)
probabilities = result

print("\n" + "="*60)
print("PREDICTION RESULTS - Risky Parlay with Emotional Betting")
print("="*60)
print(f"Loss Probability:  {probabilities[0]:.2%}")
print(f"Push Probability:  {probabilities[1]:.2%}")
print(f"Win Probability:   {probabilities[2]:.2%}")
print(f"\nPredicted Outcome: {['LOSS', 'PUSH', 'WIN'][np.argmax(probabilities)]}")
print(f"Confidence: {max(probabilities):.2%}")
print("="*60)

## 7. Save Endpoint Name for Backend Integration

In [ ]:
# Save endpoint name to use in your Next.js backend
endpoint_name = xgb_predictor.endpoint_name

print("=" * 60)
print("IMPORTANT: Save this endpoint name!")
print("=" * 60)
print(f"Endpoint Name: {endpoint_name}")
print(f"Region: {region}")
print("\nAdd these to your .env.local file:")
print(f"AWS_SAGEMAKER_ENDPOINT={endpoint_name}")
print(f"AWS_REGION={region}")
print("=" * 60)

## 8. Model Performance Metrics (Optional - requires validation data)

In [ ]:
# Evaluate on training data
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Load training data
df = pd.read_csv(local_training_file)
X = df.drop('label', axis=1).values
y_true = df['label'].values

# Get predictions (in batches to avoid timeout)
batch_size = 10
all_predictions = []

for i in range(0, len(X), batch_size):
    batch = X[i:i+batch_size]
    batch_preds = [xgb_predictor.predict(row.tolist()) for row in batch]
    all_predictions.extend(batch_preds)

# Convert to numpy array and get predicted classes
y_pred_proba = np.array(all_predictions)
y_pred = np.argmax(y_pred_proba, axis=1)

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)

print(f"\nOverall Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Loss', 'Push', 'Win']))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Loss', 'Push', 'Win'],
            yticklabels=['Loss', 'Push', 'Win'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Average probabilities per class
print("\nAverage Predicted Probabilities per True Class:")
for true_class in [0, 1, 2]:
    class_name = ['LOSS', 'PUSH', 'WIN'][true_class]
    mask = y_true == true_class
    avg_probs = y_pred_proba[mask].mean(axis=0)
    print(f"\nTrue {class_name}:")
    print(f"  Avg P(Loss): {avg_probs[0]:.2%}")
    print(f"  Avg P(Push): {avg_probs[1]:.2%}")
    print(f"  Avg P(Win):  {avg_probs[2]:.2%}")

## 9. Cleanup (Run when you're done testing)

In [ ]:
# IMPORTANT: Uncomment and run this when you want to delete the endpoint
# to avoid ongoing charges

# xgb_predictor.delete_endpoint()
# print("Endpoint deleted successfully")

## Next Steps

1. **Integrate with your backend**: Use the endpoint name to call predictions from your Next.js API
2. **Collect real data**: Replace synthetic training data with actual bet outcomes (WIN/PUSH/LOSS)
3. **Retrain regularly**: Update the model as you collect more data
4. **Monitor performance**: Track prediction accuracy and adjust hyperparameters
5. **Use probabilities**: Show users not just win/loss, but also push probability

### Backend Integration Example

See `lib/sagemaker-prediction.ts` for Node.js code to call this endpoint.